<a href="https://colab.research.google.com/github/xyle-net/VK_DPP/blob/data%2Fnsoloveva/ML(xyle_net).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install boto3 pandas scikit-learn xgboost tensorflow keras torch transformers sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

# 0. Загрузка датасета

In [ ]:
import pandas as pd
import boto3
import io

S3_BUCKET_NAME = 'xyle-net'

s3_client = boto3.client(
    "s3",
    endpoint_url="https://s3.cloud.ru",
    region_name="ru-central-1",
    aws_access_key_id="",
    aws_secret_access_key=""
)

def load_dataset():
    obj = s3_client.get_object(Bucket=S3_BUCKET_NAME, Key="data/mapping_dataset.csv")
    csv_data = obj['Body'].read().decode('utf-8')
    dataset = io.StringIO(csv_data)
    df = pd.read_csv(dataset)
    return df


# 1. TF-IDF + XGBoost (Базовый ML-классификатор)
Простой и быстрый вариант.   
Подход:
* Преобразуем текст в векторы через TfidfVectorizer.
* Обучаем XGBoost на предсказание (section_name, part_name).


In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

df = load_dataset()

# Фичи и целевые переменные
X_texts = df["content_value"]
y_section = df["section_name"]
y_part = df["part_name"]

# Кодируем section_name и part_name в числа
le_section = LabelEncoder()
le_part = LabelEncoder()
y_section_encoded = le_section.fit_transform(y_section)
y_part_encoded = le_part.fit_transform(y_part)

# TF-IDF преобразование текста
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(X_texts)

# Разделяем данные
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y_section_encoded, test_size=0.2, random_state=42)

# Обучаем XGBoost
model = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1)
model.fit(X_train, y_train)

# Оценка модели
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le_section.classes_))


              precision    recall  f1-score   support

abbreviation       0.00      0.00      0.00         1
 application       0.94      0.26      0.40       872
     content       0.80      0.10      0.18       120
  definition       0.00      0.00      0.00         1
       intro       0.78      0.33      0.47       279
        main       0.75      0.99      0.85      4316
       outro       0.79      0.09      0.16       214
      report       0.00      0.00      0.00        13
      source       0.94      0.65      0.77       535
     titular       0.93      0.38      0.54       230

    accuracy                           0.77      6581
   macro avg       0.59      0.28      0.34      6581
weighted avg       0.80      0.77      0.72      6581



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Оценка качества модели TF-IDF + XGBoost
📌 <ins>Основные наблюдения:</ins>  
✅ Общая точность (accuracy): 77% (неплохо, но есть проблемы с некоторыми классами).   
✅ Хорошие результаты для main (99% recall), source (65% recall), application (94% precision).  
❌ Плохие результаты на малочисленных классах (abbreviation, definition, report).   
❌ Macro avg F1-score = 0.34, что говорит о сильном дисбалансе классов.   
❌ Некоторые классы вообще не предсказываются (precision = 0.00).   
   
🔎 <ins>Почему модель плохо предсказывает abbreviation, definition, report?</ins>   
1. **Очень мало примеров этих классов (support = 1 или support = 13).**

Модель не обучается на таких классах → XGBoost просто игнорирует их.
Решение: Либо добавить больше данных, либо объединить редкие классы в одну категорию.  
2. **Раздел main доминирует (4316 примеров, recall = 99%)**

Модель предсказывает main почти всегда, так как он самый распространённый.
Решение: Можно уменьшить его вес с помощью class_weight или scale_pos_weight в XGBoost.
3. **TF-IDF может терять смысл текста**

Если intro и outro содержат похожие слова, модель путается.

**Решение: Попробовать Word2Vec или BERT вместо TF-IDF.**


#2. Word2Vec + LSTM (Глубокая модель с рекуррентной сетью)
LSTM (Long Short-Term Memory) учитывает порядок слов и контекст.   
Подход:
* Используем Word2Vec (или FastText) для представления текста.
* Обучаем LSTM для предсказания (section_name, part_name).

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Гиперпараметры
MAX_VOCAB_SIZE = 5000
MAX_SEQUENCE_LENGTH = 200

# Токенизация текста
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
tokenizer.fit_on_texts(df["content_value"])
X_seq = tokenizer.texts_to_sequences(df["content_value"])
X_padded = pad_sequences(X_seq, maxlen=MAX_SEQUENCE_LENGTH)

# Кодирование выходных данных
y_section_encoded = le_section.fit_transform(df["section_name"])

# Разделение данных
X_train, X_test, y_train, y_test = train_test_split(X_padded, y_section_encoded, test_size=0.2, random_state=42)

# Создаём LSTM-модель
model = Sequential([
    Embedding(MAX_VOCAB_SIZE, 128, input_length=MAX_SEQUENCE_LENGTH),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(16, activation="relu"),
    Dense(len(le_section.classes_), activation="softmax")  # Кол-во секций
])

# Компиляция и обучение
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))

# Оценка модели
y_pred = model.predict(X_test)
print(classification_report(y_test, np.argmax(y_pred, axis=1), target_names=le_section.classes_))


Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


823/823 ━━━━━━━━━━━━━━━━━━━━ 199s 235ms/step - accuracy: 0.6903 - loss: 1.1238 - val_accuracy: 0.8131 - val_loss: 0.6313
Epoch 2/5
823/823 ━━━━━━━━━━━━━━━━━━━━ 199s 232ms/step - accuracy: 0.8289 - loss: 0.5875 - val_accuracy: 0.8332 - val_loss: 0.5664
Epoch 3/5
823/823 ━━━━━━━━━━━━━━━━━━━━ 211s 242ms/step - accuracy: 0.8523 - loss: 0.4672 - val_accuracy: 0.8441 - val_loss: 0.5257
Epoch 4/5
823/823 ━━━━━━━━━━━━━━━━━━━━ 200s 242ms/step - accuracy: 0.8761 - loss: 0.3777 - val_accuracy: 0.8385 - val_loss: 0.5506
Epoch 5/5
823/823 ━━━━━━━━━━━━━━━━━━━━ 202s 243ms/step - accuracy: 0.8921 - loss: 0.3227 - val_accuracy: 0.8287 - val_loss: 0.5602
206/206 ━━━━━━━━━━━━━━━━━━━━ 12s 58ms/step
              precision    recall  f1-score   support

abbreviation       0.00      0.00      0.00         1
 application       0.73      0.76      0.74       872
     content       0.45      0.36      0.40       120
  definition       0.00      0.00      0.00         1
       intro       0.61      0.56      0.

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#3. RuBERT (Трансформеры, Hugging Face)
Самый мощный вариант: предобученная RuBERT модель.   
Подход:
* Используем RuBERT (предобученный BERT для русского языка).
* Дообучаем на наших данных для классификации section_name.

In [ ]:
import torch
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import pandas as pd

MODEL_NAME = "doc2gost/xyle-net"
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 3

# Загрузка токенизатора
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# Читаем датасет
df = pd.read_csv("dataset.csv")

# Кодируем section_name и part_name
le_section = LabelEncoder()
le_part = LabelEncoder()
df["section_encoded"] = le_section.fit_transform(df["section_name"])
df["part_encoded"] = le_part.fit_transform(df["part_name"])

# Разделение данных
train_texts, test_texts, train_sections, test_sections, train_parts, test_parts = train_test_split(
    df["content_value"], df["section_encoded"], df["part_encoded"], test_size=0.2, random_state=42
)

# Создаём PyTorch Dataset
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

# Датасеты для section_name
train_dataset_section = TextDataset(train_texts.tolist(), train_sections.tolist(), tokenizer, MAX_LENGTH)
test_dataset_section = TextDataset(test_texts.tolist(), test_sections.tolist(), tokenizer, MAX_LENGTH)

# Датасеты для part_name
train_dataset_part = TextDataset(train_texts.tolist(), train_parts.tolist(), tokenizer, MAX_LENGTH)
test_dataset_part = TextDataset(test_texts.tolist(), test_parts.tolist(), tokenizer, MAX_LENGTH)

# DataLoader
train_loader_section = DataLoader(train_dataset_section, batch_size=BATCH_SIZE, shuffle=True)
test_loader_section = DataLoader(test_dataset_section, batch_size=BATCH_SIZE)

train_loader_part = DataLoader(train_dataset_part, batch_size=BATCH_SIZE, shuffle=True)
test_loader_part = DataLoader(test_dataset_part, batch_size=BATCH_SIZE)

# Загружаем две модели (одна для section, другая для part)
model_section = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(le_section.classes_))
model_part = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(le_part.classes_))

device = "cuda" if torch.cuda.is_available() else "cpu"
model_section.to(device)
model_part.to(device)

optimizer_section = torch.optim.AdamW(model_section.parameters(), lr=2e-5)
optimizer_part = torch.optim.AdamW(model_part.parameters(), lr=2e-5)

# Функция для обучения модели
def train_model(model, optimizer, train_loader, epochs):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch + 1}, Loss: {total_loss / len(train_loader)}")

# Обучение section_name
print("🔹 Обучение модели для section_name...")
train_model(model_section, optimizer_section, train_loader_section, EPOCHS)

# Обучение part_name
print("🔹 Обучение модели для part_name...")
train_model(model_part, optimizer_part, train_loader_part, EPOCHS)

print("✅ Обучение завершено!")

# Функция для оценки модели
def evaluate_model(model, test_loader, label_encoder):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].cpu().numpy()

            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels)

    accuracy = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=label_encoder.classes_, digits=4)

    return accuracy, report

# Оценка модели section_name
print("📊 Оценка модели section_name...")
accuracy_section, report_section = evaluate_model(model_section, test_loader_section, le_section)
print(f"✅ Accuracy: {accuracy_section:.4f}")
print(report_section)

# Оценка модели part_name
print("📊 Оценка модели part_name...")
accuracy_part, report_part = evaluate_model(model_part, test_loader_part, le_part)
print(f"✅ Accuracy: {accuracy_part:.4f}")
print(report_part)
